In [4]:
import pandas as pd

raw = pd.read_csv(
    "nelder_mead_raw.csv"
)

good = raw[raw["success"] == True]

print(
    good["joint_gamma_est"].describe()
)

print(
    "Mean gamma:",
    good["joint_gamma_est"].mean()
)

print(
    "Median gamma:",
    good["joint_gamma_est"].median()
)

print(
    "Bias:",
    good["joint_gamma_est"].mean() - 1.17
)

count    720.000000
mean      -1.491817
std        2.462604
min       -8.734065
25%       -2.829421
50%       -1.753320
75%       -0.357849
max       34.422443
Name: joint_gamma_est, dtype: float64
Mean gamma: -1.4918165263246694
Median gamma: -1.7533204714363912
Bias: -2.661816526324669


In [5]:
raw = raw[raw["alpha_true"] != 7.7].copy()

In [6]:
gamma_recovery = (
    raw[raw["success"] == True]
    .groupby(["alpha_true", "gamma_true"])
    .agg(
        n=("rep", "size"),
        mean_gamma_est=("joint_gamma_est", "mean"),
        median_gamma_est=("joint_gamma_est", "median"),
        sd_gamma_est=("joint_gamma_est", "std"),
    )
    .reset_index()
)

gamma_recovery["bias"] = (
    gamma_recovery["mean_gamma_est"]
    - gamma_recovery["gamma_true"]
)

print(gamma_recovery.round(3))

   alpha_true  gamma_true   n  mean_gamma_est  median_gamma_est  sd_gamma_est  \
0       -7.70       -2.62  60          -1.009            -1.361         3.518   
1       -7.70        0.00  60          -1.524            -1.493         1.300   
2       -7.70        1.23  60          -1.982            -1.950         1.674   
3       -4.30       -2.62  60          -1.572            -1.684         1.847   
4       -4.30        0.00  60          -1.221            -2.102         4.886   
5       -4.30        1.23  60          -1.373            -1.420         1.771   
6       -1.66       -2.62  60          -3.358            -3.219         1.280   
7       -1.66        0.00  60          -1.266            -0.975         1.402   
8       -1.66        1.23  60          -1.235            -1.523         1.812   

    bias  
0  1.611  
1 -1.524  
2 -3.212  
3  1.048  
4 -1.221  
5 -2.603  
6 -0.738  
7 -1.266  
8 -2.465  


In [7]:
def summarize_raw(raw, label, particle_counts=(500, 1000)):

    good = raw[raw["success"] == True].copy()

    keys = [
        "mu_true",
        "alpha_true",
        "gamma_true",
        "phi_true",
        "r_true"
    ]

    rows = []

    for N in particle_counts:

        out = (
            good
            .groupby(keys)
            .agg(
                n=("rep", "size"),

                mean_delta_nll=(f"delta_nll_{N}", "mean"),
                median_delta_nll=(f"delta_nll_{N}", "median"),
                joint_nll_win_rate=(f"joint_wins_nll_{N}", "mean"),

                mean_delta_bic=(f"delta_bic_{N}", "mean"),
                median_delta_bic=(f"delta_bic_{N}", "median"),
                joint_bic_win_rate=(f"joint_wins_bic_{N}", "mean"),

                joint_convergence_rate=(
                    "joint_optimizer_success", "mean"
                ),

                gamma0_convergence_rate=(
                    "gamma0_optimizer_success", "mean"
                ),

                mean_gamma_est=(
                    "joint_gamma_est", "mean"
                ),

                median_gamma_est=(
                    "joint_gamma_est", "median"
                ),

                sd_gamma_est=(
                    "joint_gamma_est", "std"
                ),
            )
            .reset_index()
        )

        out["gamma_bias"] = (
            out["mean_gamma_est"]
            - out["gamma_true"]
        )

        out["particles"] = N
        out["run"] = label

        rows.append(out)

    return pd.concat(
        rows,
        ignore_index=True
    )

In [8]:
s1 = summarize_raw(raw, "run1")

In [9]:
s1[
    [
        "alpha_true",
        "gamma_true",
        "particles",
        "mean_delta_nll",
        "median_delta_nll",
        "joint_nll_win_rate",
        "mean_delta_bic",
        "median_delta_bic",
        "joint_bic_win_rate",
        "gamma_bias",
        "joint_convergence_rate",
        "gamma0_convergence_rate",
    ]
].round(3)

,alpha_true,gamma_true,particles,mean_delta_nll,median_delta_nll,joint_nll_win_rate,mean_delta_bic,median_delta_bic,joint_bic_win_rate,gamma_bias,joint_convergence_rate,gamma0_convergence_rate
0,-7.70,-2.62,500,0.124,0.063,0.60,-3.846,-3.968,0.00,1.162,1.00,1.0
1,-7.70,-2.62,500,3.924,0.205,0.75,3.754,-3.685,0.05,2.008,1.00,1.0
2,-7.70,-2.62,500,0.269,0.256,0.75,-3.557,-3.582,0.00,1.662,1.00,1.0
3,-7.70,0.00,500,0.264,0.107,0.60,-3.566,-3.881,0.00,-1.382,1.00,1.0
4,-7.70,0.00,500,-0.020,-0.059,0.45,-4.135,-4.212,0.00,-1.410,1.00,1.0
5,-7.70,0.00,500,0.183,-0.314,0.35,-3.728,-4.722,0.10,-1.781,1.00,1.0
6,-7.70,1.23,500,0.060,0.045,0.55,-3.974,-4.005,0.00,-3.309,1.00,1.0
7,-7.70,1.23,500,26.617,0.131,0.55,49.139,-3.832,0.05,-2.901,1.00,1.0
8,-7.70,1.23,500,0.253,0.219,0.60,-3.589,-3.657,0.00,-3.427,1.00,1.0
9,-4.30,-2.62,500,0.362,0.071,0.60,-3.371,-3.952,0.05,1.062,1.00,1.0
